# 04 · Recall — 6bba FN teşhisi + instance ayrımı

## Neden burası
Baseline edge_J **0.78'de takılı** (LB 0.749). Kaybın (**468 FN**) **%91'i** iki büyük 6bba
dataset'inde (recall 0.73 / 0.88). `jaccard ≈ recall²` etkisiyle bu, skoru doğrudan tavana bağlıyor.

## Hipotez
Çekirdekler **temas halinde** (EDA: yerel kontrast ~1.5×). Tek Otsu eşiği + **intensity local-max**:
iki bitişik çekirdekte parlak olan sönüğü **bastırıyor** → tek tepe → sönük çekirdek kaçıyor.

## Fikir: şekil-tabanlı ayrım (watershed)
İki temas eden çekirdek intensity'de tek tepe verse de **distance transform**'da iki ayrı tepe
verir (her çekirdeğin merkezi kendi kütlesinin en derini). Klasik nuclei-segmentation.

## Plan (tracking YOK — hızlı, RAM'de)
1. Teşhis: 6bba kaçan GT çevresi → temas mı, sönük mü?
2. `intensity-localmax` (mevcut) vs `watershed` vs `hmaxima` → recall@7µm + yoğunluk
3. En iyiyi tam metrikle (edge_J) doğrula

## 0 · Kurulum

In [ ]:
import sys, subprocess, os, time
def _scan(base):
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if not d.endswith((".zarr",".geff")) and d!="competitions"]
        if root.count(os.sep) > 9: dirs[:]=[]; continue
        yield root, files
def ensure_zarr():
    try:
        import zarr; return zarr
    except ImportError: pass
    for root, files in _scan("/kaggle/input"):
        if os.path.basename(root)=="zarr" and "__init__.py" in files:
            p=os.path.dirname(root); sys.path.insert(0,p)
            try:
                import zarr; print("zarr <- sys.path:",p); return zarr
            except ImportError: sys.path.pop(0)
    subprocess.run([sys.executable,"-m","pip","install","-q","zarr"],check=False)
    import zarr; return zarr
zarr=ensure_zarr(); print("zarr:",zarr.__version__)

import numpy as np, pandas as pd
from pathlib import Path
from collections import Counter
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from skimage.filters import threshold_otsu
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.morphology import h_maxima
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

SCALE=(1.625,0.40625,0.40625); S=np.array(SCALE,dtype=np.float32)
MATCH_UM=7.0
FIG=Path("/kaggle/working/figures"); FIG.mkdir(parents=True,exist_ok=True)
print("hazir")

In [ ]:
INPUT=Path("/kaggle/input")
def find_root():
    st=[(INPUT,0)]
    while st:
        b,d=st.pop()
        try:
            if (b/"train").is_dir() and (b/"test").is_dir(): return b
        except Exception: pass
        if d<4:
            for c in sorted(b.iterdir()):
                if c.is_dir() and not c.name.endswith((".zarr",".geff")): st.append((c,d+1))
ROOT=find_root(); TRAIN=ROOT/"train"; TEST=ROOT/"test"
test_names=sorted(p.stem for p in TEST.glob("*.zarr"))
def open_image(zp):
    n=zarr.open(str(zp),mode="r"); a=dict(n.attrs)
    ms=a.get("multiscales") or (a.get("ome") or {}).get("multiscales")
    if ms: return n[ms[0]["datasets"][0]["path"]]
    return n["0"] if "0" in list(n.keys()) else n
def load_geff(gp):
    g=zarr.open(str(gp),mode="r"); nodes=g["nodes"]; ids=np.asarray(nodes["ids"]); props={}
    for pn in list(nodes["props"].keys()):
        try: props[pn]=np.asarray(nodes["props"][pn]["values"])
        except Exception: pass
    d={"id":ids}
    for k in ("t","z","y","x"):
        if k in props: d[k]=props[k]
    return pd.DataFrame(d), np.asarray(g["edges"]["ids"])
print("test:",test_names)

## 1 · Kareleri yükle (adaptif: seyrek dataset'ten çok kare)

In [ ]:
TARGET_GT=45; MAX_FR=14
FRAMES=[]; GT_ALL={}
t0=time.time()
for nm in test_names:
    gdf,ge=load_geff(TRAIN/(nm+".geff")); GT_ALL[nm]=(gdf,ge)
    arr=open_image(TEST/(nm+".zarr"))
    cnt=gdf.groupby("t").size().sort_values(ascending=False)
    picks=[]; tot=0
    for t,c in cnt.items():
        picks.append(int(t)); tot+=int(c)
        if tot>=TARGET_GT or len(picks)>=MAX_FR: break
    for t in picks:
        FRAMES.append((nm,t,np.asarray(arr[t]).astype(np.float32),
                       gdf[gdf.t==t][["z","y","x"]].values.astype(np.float32)))
print(f"{len(FRAMES)} kare, {time.time()-t0:.0f}s, RAM ~{sum(f[2].nbytes for f in FRAMES)/1e6:.0f} MB")
for nm in test_names:
    fr=[f for f in FRAMES if f[0]==nm]
    print(f"  {nm}: {len(fr)} kare, {sum(len(f[3]) for f in fr)} GT")

## 2 · Detection yöntemleri

In [ ]:
def m_intensity(v, sigma=(1,2,2), foot=(3,11,11), k=1.0):
    # MEVCUT baseline: intensity yumusatma -> Otsu -> local-max
    sm=ndi.gaussian_filter(v, sigma=sigma); thr=threshold_otsu(sm)*k
    mx=ndi.maximum_filter(sm, size=foot); peaks=(sm==mx)&(sm>thr)
    lbl,n=ndi.label(peaks)
    if n==0: return np.zeros((0,3),np.float32)
    return np.asarray(ndi.center_of_mass(sm,lbl,np.arange(1,n+1)),np.float32)

def m_watershed(v, sigma=(1,2,2), k=1.0, min_dist_um=4.0):
    # DISTANCE-tabanli: foreground -> anizotropik distance -> distance tepeleri
    sm=ndi.gaussian_filter(v, sigma=sigma); thr=threshold_otsu(sm)*k
    fg=sm>thr
    if not fg.any(): return np.zeros((0,3),np.float32)
    dist=ndi.distance_transform_edt(fg, sampling=SCALE)   # um
    # min ayrim: um -> voxel footprint
    fp=tuple(int(max(1,round(min_dist_um/s))) for s in SCALE)
    coords=peak_local_max(dist, footprint=np.ones(tuple(2*f+1 for f in fp)),
                          labels=fg, exclude_border=False)
    return coords.astype(np.float32) if len(coords) else np.zeros((0,3),np.float32)

def m_hmaxima(v, sigma=(1,2,2), h_frac=0.10, foot=(3,11,11)):
    # h-maxima: >h yukselen anlamli tepeler (kucuk gurultu tepelerini birlestirir)
    sm=ndi.gaussian_filter(v, sigma=sigma)
    h=h_frac*(sm.max()-sm.min())
    hm=h_maxima(sm, h)                    # ikili tepe bolgeleri
    lbl,n=ndi.label(hm)
    if n==0: return np.zeros((0,3),np.float32)
    return np.asarray(ndi.center_of_mass(sm,lbl,np.arange(1,n+1)),np.float32)

def recall_dens(cents, gt):
    if len(cents)==0 or len(gt)==0: return (0.0 if len(gt) else 1.0), len(cents)
    return float((cdist(gt*S,cents*S).min(1)<=MATCH_UM).mean()), len(cents)
print("ok")

## 3 · Teşhis: 6bba'da kaçan GT — temas mı, sönük mü?
Mevcut yöntemle kaçan GT'lerin çevresini göster + o konumdaki intensity vs local çevre.

In [ ]:
BAD="6bba_05b6850b"
shown=0; miss_bright=[]; miss_isbright=[]
fig,axes=plt.subplots(2,3,figsize=(15,9)); axes=axes.ravel()
for nm,t,v,gp in FRAMES:
    if nm!=BAD: continue
    c=m_intensity(v)
    D=cdist(gp*S,c*S) if len(c) else np.full((len(gp),1),1e9)
    for i,p in enumerate(gp):
        if D[i].min()<=MATCH_UM: continue          # kacirilanlar
        z,y,x=int(round(p[0])),int(round(p[1])),int(round(p[2]))
        # GT parlakligi lokal cevreye gore yuksek mi? (sonuk mu, yoksa bastirilmis mi?)
        z0,z1=max(0,z-1),min(v.shape[0],z+2); y0,y1=max(0,y-10),min(v.shape[1],y+11); x0,x1=max(0,x-10),min(v.shape[2],x+11)
        loc=v[z0:z1,y0:y1,x0:x1]
        miss_isbright.append(v[z,y,x] > np.percentile(loc,75))   # lokalde parlak mi
        if shown<6:
            a=axes[shown]; a.imshow(v[z,max(0,y-30):y+30,max(0,x-30):x+30],cmap="gray")
            a.plot(min(30,x),min(30,y),"rx",ms=14,mew=3)
            for q in c[np.abs(c[:,0]-z)<=2]:
                if abs(q[1]-y)<30 and abs(q[2]-x)<30: a.plot(q[2]-max(0,x-30),q[1]-max(0,y-30),"g+",ms=9,mew=2)
            a.set_title(f"t={t} | GT val={v[z,y,x]:.0f} lokal-p90={np.percentile(loc,90):.0f}",fontsize=8); a.axis("off"); shown+=1
for j in range(shown,6): axes[j].axis("off")
plt.suptitle(f"{BAD} kaçan GT (kırmızı x) + tespitler (yeşil +)")
plt.tight_layout(); plt.savefig(FIG/"D14_missed_6bba.png",dpi=120,bbox_inches="tight"); plt.show()
if miss_isbright:
    frac=np.mean(miss_isbright)
    print(f"\nKACAN GT'lerin %{100*frac:.0f}'i lokal cevrede PARLAK (>p75).")
    print(">> Yuksekse: cekirdek gorunur ama local-max degil (INSTANCE/BASTIRMA sorunu -> watershed).")
    print(">> Dusukse: cekirdek sonuk (ESIK sorunu -> daha dusuk esik).")

## 4 · Yöntem karşılaştırması — havuzlanmış recall + yoğunluk
Dataset başına ayrı (44b6 vs 6bba farklı davranabilir). Yoğunluğu T_true ile kıyasla.

In [ ]:
T_TRUE={}
for nm in test_names:
    g=zarr.open(str(TRAIN/(nm+".geff")),mode="r")
    T_TRUE[nm]=dict(g.attrs).get("geff",{}).get("extra",{}).get("estimated_number_of_nodes")

METHODS={
    "intensity(mevcut)": lambda v: m_intensity(v),
    "watershed d=4um":   lambda v: m_watershed(v, min_dist_um=4.0),
    "watershed d=5um":   lambda v: m_watershed(v, min_dist_um=5.0),
    "watershed d=6um":   lambda v: m_watershed(v, min_dist_um=6.0),
    "hmaxima h=0.10":    lambda v: m_hmaxima(v, h_frac=0.10),
}
rows=[]; t0=time.time()
for name,fn in METHODS.items():
    per={nm:{"m":0,"g":0,"d":[]} for nm in test_names}
    for nm,t,v,gp in FRAMES:
        c=fn(v); rec,den=recall_dens(c,gp)
        per[nm]["m"]+=int(round(rec*len(gp))); per[nm]["g"]+=len(gp); per[nm]["d"].append(den)
    M=sum(p["m"] for p in per.values()); G=sum(p["g"] for p in per.values())
    r=dict(method=name, recall=round(M/max(G,1),3))
    for nm in test_names:
        p=per[nm]; r[nm[:9]]=round(p["m"]/max(p["g"],1),2)
        r[nm[:9]+"_x"]=round(np.mean(p["d"])*100/T_TRUE[nm],2)   # T_pred/T_true
    rows.append(r); print(f"{name:20s} recall={r['recall']:.3f}")
res=pd.DataFrame(rows).sort_values("recall",ascending=False)
print(f"\n{time.time()-t0:.0f}s\n=== KARSILASTIRMA (recall + dataset recall + T_pred/T_true) ===")
print(res.to_string(index=False))
print("\n(_x = T_pred/T_true; 1.0 ideal. recall yuksek ama _x>>1 ise fazla-tahmin/linking riski)")

## 5 · En iyi yöntemi tam metrikle doğrula (opsiyonel, ~5dk/yöntem)
Recall vekil ölçüt; asıl edge_J linking'i de içerir (v2'de recall↑ ama jaccard↓ görmüştük).
Sadece recall'da mevcut'u geçen yöntem(ler) için koş.

In [ ]:
LINK_MAX_UM=8.0
def link_pairs(A,B):
    if len(A)==0 or len(B)==0: return []
    D=cdist(A*S,B*S); cost=np.where(D<=LINK_MAX_UM,D,1e6)
    r,c=linear_sum_assignment(cost)
    return [(int(i),int(j)) for i,j in zip(r,c) if D[i,j]<=LINK_MAX_UM]
def track_eval(nm, method_fn):
    arr=open_image(TEST/(nm+".zarr")); T=arr.shape[0]
    cents=[method_fn(np.asarray(arr[t]).astype(np.float32)) for t in range(T)]
    nodes=[]; edges=[]; off=[]; nid=1
    for t,c in enumerate(cents):
        off.append(nid)
        for p in c: nodes.append((nid,t)); nid+=1
    coords={}
    idn=1
    for t,c in enumerate(cents):
        for p in c: coords[idn]=(t,p); idn+=1
    for t in range(T-1):
        for i,j in link_pairs(cents[t],cents[t+1]): edges.append((off[t]+i,off[t+1]+j))
    # eval
    gdf,ge=GT_ALL[nm]
    pn=pd.DataFrame([(k,)+ (coords[k][0],)+tuple(coords[k][1]) for k in coords],
                    columns=["node_id","t","z","y","x"])
    gmap={}
    for t,g in gdf.groupby("t"):
        p=pn[pn.t==int(t)]
        if len(p)==0 or len(g)==0: continue
        D=cdist(p[["z","y","x"]].values*S,g[["z","y","x"]].values*S)
        cost=np.where(D<=MATCH_UM,D,1e6); r,c=linear_sum_assignment(cost)
        pid=p["node_id"].values; gid=g["id"].values
        for i,j in zip(r,c):
            if D[i,j]<=MATCH_UM: gmap[int(pid[i])]=int(gid[j])
    gtset=set((int(u),int(v)) for u,v in ge); TP=0;FP=0;cov=set()
    for u,v in edges:
        gu=gmap.get(u); gv=gmap.get(v)
        if gu is None or gv is None: continue
        if (gu,gv) in gtset: TP+=1;cov.add((gu,gv))
        else: FP+=1
    return TP,FP,len(gtset)-len(cov),len(nodes)

RUN_FULL=True
CAND=["watershed d=5um","intensity(mevcut)"]   # elle: en umutlu + kontrol
if RUN_FULL:
    for name in CAND:
        fn=METHODS[name]; t0=time.time(); eTP=eFP=eFN=nn=0; per=[]
        for nm in test_names:
            tp,fp,fn_,nnn=track_eval(nm,fn); eTP+=tp;eFP+=fp;eFN+=fn_;nn+=nnn
            per.append((nm,tp,fp,fn_))
        J=eTP/max(eTP+eFP+eFN,1)
        print(f"\n=== {name} ===")
        for nm,tp,fp,fn_ in per: print(f"  {nm}: TP={tp} FP={fp} FN={fn_}")
        print(f"  >> MICRO edge_J={J:.4f} [TP={eTP} FP={eFP} FN={eFN}] node/kare~{nn/100/len(test_names):.0f} ({time.time()-t0:.0f}s)")
    print("\n(referans: baseline edge_J=0.7796, LB=0.749)")

## 6 · Karar
- [ ] Teşhis: kaçan GT parlak mı (instance) yoksa sönük mü (eşik)?
- [ ] En iyi yöntem + recall = **…**, edge_J = **…** (baseline 0.7796)
- [ ] Yoğunluk (T_pred/T_true) makul mü?
- [ ] Kazandıysa `02_baseline`'a taşı → submit